# Adversarial Robustness in Deep Learning for Image Classification

This Google Colab notebook runs a complete PyTorch project for CIFAR-10 image classification with clean training, FGSM/PGD attacks, PGD-based adversarial training, robustness evaluation, and visualization.

## Colab Workflow

1. Upload or clone the `adv_project` folder into `/content`.
2. Enable GPU from `Runtime -> Change runtime type -> GPU`.
3. Run the cells in order.

The notebook automatically creates checkpoints, logs, plots, and metric files.

In [ ]:
from pathlib import Path
import os
import sys

import torch


def find_project_root() -> Path:
    candidates = [Path('/content/adv_project'), Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'utils' / 'config.py').exists():
            return candidate
        if (candidate / 'adv_project' / 'utils' / 'config.py').exists():
            return candidate / 'adv_project'
    raise FileNotFoundError('Upload or clone the adv_project folder into /content before running this notebook.')


PROJECT_ROOT = find_project_root().resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_dirs = [
    PROJECT_ROOT / 'checkpoints',
    PROJECT_ROOT / 'outputs',
    PROJECT_ROOT / 'outputs' / 'logs',
    PROJECT_ROOT / 'outputs' / 'plots',
    PROJECT_ROOT / 'outputs' / 'metrics',
]
for directory in required_dirs:
    directory.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')


In [ ]:
import json
from functools import partial

import pandas as pd

from attacks.pgd import pgd_attack
from evaluation.robustness import accuracy_vs_epsilon, robustness_suite
from evaluation.visualization import plot_robustness_curves, plot_training_curves, visualize_adversarial_examples
from models.cnn import CustomCNN
from models.resnet import build_resnet18
from training.adversarial_training import PGDAdversarialTrainer
from training.dataloader import get_cifar10_dataloaders
from training.trainer import Trainer, build_criterion, build_optimizer_and_scheduler
from utils.config import build_config, get_device
from utils.logger import MetricLogger, setup_logger

config = build_config(PROJECT_ROOT)
config.experiment.train_epochs = 5
config.experiment.adv_train_epochs = 5
config.data.batch_size = 128
config.data.num_workers = 2
config.attack.pgd_steps = 7
config.save(PROJECT_ROOT / 'outputs' / 'run_config.json')

device = get_device(config)
print(f'Using device: {device}')
print(json.dumps(config.to_dict(), indent=2))


In [ ]:
train_loader, val_loader, test_loader, class_names = get_cifar10_dataloaders(config)

print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')
print(f'Classes: {class_names}')


In [ ]:
def create_model(model_name: str):
    if model_name == 'cnn':
        return CustomCNN(num_classes=config.experiment.num_classes)
    if model_name == 'resnet18':
        return build_resnet18(num_classes=config.experiment.num_classes, use_pretrained=False)
    raise ValueError(f'Unsupported model: {model_name}')


def create_standard_trainer(model, experiment_name: str):
    optimizer, scheduler = build_optimizer_and_scheduler(model, config, config.experiment.train_epochs)
    logger = setup_logger('trainer', PROJECT_ROOT / 'outputs' / 'logs', experiment_name)
    metric_logger = MetricLogger(PROJECT_ROOT / 'outputs' / 'metrics' / f'{experiment_name}.csv')
    return Trainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=build_criterion(),
        device=device,
        checkpoint_dir=PROJECT_ROOT / 'checkpoints',
        logger=logger,
        metric_logger=metric_logger,
        use_amp=config.experiment.use_amp,
    )


def create_adversarial_trainer(model, experiment_name: str):
    optimizer, scheduler = build_optimizer_and_scheduler(model, config, config.experiment.adv_train_epochs)
    logger = setup_logger('adv_trainer', PROJECT_ROOT / 'outputs' / 'logs', experiment_name)
    metric_logger = MetricLogger(PROJECT_ROOT / 'outputs' / 'metrics' / f'{experiment_name}.csv')
    return PGDAdversarialTrainer(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=build_criterion(),
        device=device,
        checkpoint_dir=PROJECT_ROOT / 'checkpoints',
        mean=config.data.mean,
        std=config.data.std,
        epsilon=config.attack.pgd_epsilon,
        alpha=config.attack.pgd_alpha,
        steps=config.attack.pgd_steps,
        logger=logger,
        metric_logger=metric_logger,
        use_amp=config.experiment.use_amp,
    )


In [ ]:
trained_models = {}
histories = {}

for model_name in config.experiment.run_models:
    experiment_name = f'{model_name}_standard'
    print(f'Training {experiment_name}...')
    model = create_model(model_name).to(device)
    trainer = create_standard_trainer(model, experiment_name)
    history = trainer.fit(train_loader, val_loader, config.experiment.train_epochs, experiment_name)
    trained_models[experiment_name] = model
    histories[experiment_name] = history


In [ ]:
for model_name in config.experiment.adv_models:
    experiment_name = f'{model_name}_adv_pgd'
    print(f'Adversarial training {experiment_name}...')
    model = create_model(model_name).to(device)
    trainer = create_adversarial_trainer(model, experiment_name)
    history = trainer.fit(train_loader, val_loader, config.experiment.adv_train_epochs, experiment_name)
    trained_models[experiment_name] = model
    histories[experiment_name] = history


In [ ]:
robustness_results = {}
for experiment_name, model in trained_models.items():
    robustness_results[experiment_name] = robustness_suite(model, test_loader, device, config)

results_df = pd.DataFrame(robustness_results).T
results_df = results_df[[
    'clean_accuracy',
    'fgsm_accuracy',
    'pgd_accuracy',
    'clean_loss',
    'fgsm_loss',
    'pgd_loss',
]]
results_df = results_df.sort_values('pgd_accuracy', ascending=False)
results_df.to_csv(PROJECT_ROOT / 'outputs' / 'metrics' / 'robustness_summary.csv')
results_df


In [ ]:
training_curve_path = PROJECT_ROOT / 'outputs' / 'plots' / 'training_curves.png'
plot_training_curves(histories, training_curve_path, title='Training Curves: Standard vs PGD Adversarial Training')
print(f'Saved training curve plot to: {training_curve_path}')


In [ ]:
fgsm_curves = {}
pgd_curves = {}

for experiment_name, model in trained_models.items():
    fgsm_curves[experiment_name] = accuracy_vs_epsilon(
        model,
        test_loader,
        device,
        attack_name='fgsm',
        epsilons=config.attack.fgsm_curve_epsilons,
        mean=config.data.mean,
        std=config.data.std,
    )
    pgd_curves[experiment_name] = accuracy_vs_epsilon(
        model,
        test_loader,
        device,
        attack_name='pgd',
        epsilons=config.attack.fgsm_curve_epsilons,
        mean=config.data.mean,
        std=config.data.std,
        alpha=config.attack.pgd_alpha,
        steps=config.attack.pgd_steps,
    )

fgsm_curve_path = PROJECT_ROOT / 'outputs' / 'plots' / 'fgsm_accuracy_vs_epsilon.png'
pgd_curve_path = PROJECT_ROOT / 'outputs' / 'plots' / 'pgd_accuracy_vs_epsilon.png'

plot_robustness_curves(fgsm_curves, fgsm_curve_path, title='FGSM Robustness Comparison')
plot_robustness_curves(pgd_curves, pgd_curve_path, title='PGD Robustness Comparison')

print(f'Saved FGSM curve to: {fgsm_curve_path}')
print(f'Saved PGD curve to: {pgd_curve_path}')


In [ ]:
reference_experiment = 'resnet18_adv_pgd' if 'resnet18_adv_pgd' in trained_models else next(iter(trained_models))
attack_fn = partial(
    pgd_attack,
    epsilon=config.attack.pgd_epsilon,
    alpha=config.attack.pgd_alpha,
    steps=config.attack.pgd_steps,
    mean=config.data.mean,
    std=config.data.std,
    random_start=True,
)
examples_path = PROJECT_ROOT / 'outputs' / 'plots' / f'{reference_experiment}_adversarial_examples.png'
visualize_adversarial_examples(
    trained_models[reference_experiment],
    test_loader,
    device,
    attack_fn,
    class_names,
    config.data.mean,
    config.data.std,
    examples_path,
    num_images=6,
)
print(f'Saved adversarial example comparison to: {examples_path}')


In [ ]:
checkpoint_files = sorted(str(path.relative_to(PROJECT_ROOT)) for path in (PROJECT_ROOT / 'checkpoints').glob('*.pth'))
output_files = sorted(
    str(path.relative_to(PROJECT_ROOT))
    for path in (PROJECT_ROOT / 'outputs').rglob('*')
    if path.is_file()
)

print('Saved checkpoints:')
for file_name in checkpoint_files:
    print(f'  - {file_name}')

print('\nSaved outputs:')
for file_name in output_files:
    print(f'  - {file_name}')
